In [1]:
import os
import torch
import mlflow
import dagshub
from ultralytics import settings
settings.update({'mlflow': False})
from ultralytics import YOLO

In [2]:
torch.cuda.empty_cache()

In [3]:
model_path = 'downloads/best.pt'
model = YOLO(model_path)

In [4]:
model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
try:
    _, params, _, flops = model.info(verbose=False)
except Exception:
    params = sum(p.numel() for p in model.model.parameters())
    flops = 0.0

In [ ]:
metrics = model.val(
    data='data.yaml',
    device=0,       
    batch=4,
    imgsz=640,
    split='val',
    project='PPE_Detection',
    name='baseline_evaluation'
)

# Trích xuất dữ liệu
map50 = metrics.box.map50
map50_95 = metrics.box.map
precision = metrics.box.mp
recall = metrics.box.mr
f1_score = 2 * (precision * recall) / (precision + recall + 1e-16)
inference_time = metrics.speed['inference']
save_dir = metrics.save_dir

print("\nĐánh giá xong! Bắt đầu quá trình đồng bộ lên DagsHub...")

WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.104  Python-3.12.7 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
Model summary (fused): 93 layers, 25,844,971 parameters, 0 gradients, 78.7 GFLOPs
WARNING val: Slow image access detected (ping: 0.40.2 ms, read: 4.52.3 MB/s, size: 71.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\ppe_cv\train_data\valid\labels.cache... 229 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 229/229  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 58/58 7.6it/s 7.6s<0.1s
                   all        229       2019      0.742      0.689      0.714      0.355
                 boots         93        201      0.582      0.567      0.546      0.228
                gloves         78        157       0

In [6]:
MY_USERNAME = "TQanh23" 
REPO_NAME = "ppe_v2"

In [7]:
dagshub.init(repo_owner=MY_USERNAME, repo_name=REPO_NAME, mlflow=True)
mlflow.set_experiment("PPE__yolov8m_Baseline_Evaluation")

Accessing as TQanh23

Initialized MLflow to track repo "TQanh23/ppe_v2"

Repository TQanh23/ppe_v2 initialized!

<Experiment: artifact_location='mlflow-artifacts:/5ca5f51ddfd64c5d93284ad770344737', creation_time=1784964995164, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1784964995164, lifecycle_stage='active', name='PPE__yolov8m_Baseline_Evaluation', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [8]:
if mlflow.active_run():
    mlflow.end_run()

In [ ]:
with mlflow.start_run(run_name="YOLOv8m_150e_Baseline_Val_2") as run:
    mlflow.log_params({
        "model_type": "YOLOv8m",
        "imgsz": 640,
        "batch_size": 4,
        "parameters": params,
        "FLOPs_G": flops,
        "model_size_MB": round(model_size_mb, 2)
    })
    
    mlflow.log_metrics({
        "mAP_50": map50,
        "mAP_50_95": map50_95,
        "Precision": precision,
        "Recall": recall,
        "F1_score": f1_score,
        "Inference_Time_ms": inference_time
    })
    
    mlflow.log_artifact(model_path, artifact_path="models")
    mlflow.log_artifacts(save_dir, artifact_path="validation_graphs")

    print("\n" + "="*50)
    print(" ĐÃ ĐỒNG BỘ THÀNH CÔNG ".center(50))
    print("="*50)
    print(f"Run ID:                  {run.info.run_id}")
    print(f"mAP@50:                  {map50:.4f}")
    print(f"F1-score:                {f1_score:.4f}")
    print(f"Thời gian suy luận:      {inference_time:.2f} ms/ảnh")
    print("="*50)


              ĐÃ ĐỒNG BỘ THÀNH CÔNG               
Run ID:                  a7de2feb405b4e1b8d19b347ef7e247c
mAP@50:                  0.7137
F1-score:                0.7146
Thời gian suy luận:      16.30 ms/ảnh
🏃 View run YOLOv8m_150e_Baseline_Val at: https://dagshub.com/TQanh23/ppe_v2.mlflow/#/experiments/4/runs/a7de2feb405b4e1b8d19b347ef7e247c
🧪 View experiment at: https://dagshub.com/TQanh23/ppe_v2.mlflow/#/experiments/4
